<a href="https://colab.research.google.com/github/diego-ascenciorod/Clase-Laboratorio-Estadistico/blob/main/A05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay

# cargar dataset
df = pd.read_csv("Heart Prediction Quantum Dataset.csv")

# ver datos generales
print("primeras filas")
display(df.head())

print("shape:", df.shape)

print("tipos de datos")
print(df.dtypes)

print("valores nulos")
print(df.isnull().sum())

print("variable objetivo")
print(df["HeartDisease"].value_counts())

# resumen estadistico
print("resumen estadistico")
display(df.describe())

# histogramas
print("histogramas")
df.hist(figsize=(10,6))
plt.show()

# correlacion
print("matriz de correlacion")
corr = df.corr(numeric_only=True)
display(corr)

plt.figure(figsize=(8,6))
plt.imshow(corr, cmap="coolwarm")
plt.colorbar()
plt.title("matriz de correlacion")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.show()

# definir variables
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# separar variables numericas y categoricas
num = ["Age", "BloodPressure", "Cholesterol", "HeartRate", "QuantumPatternFeature"]
cat = ["Gender"]

# dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# preprocesamiento
pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", "passthrough", cat)
])

# modelo
model = MLPClassifier(hidden_layer_sizes=(5,10), random_state=42, max_iter=2000)

# pipeline
pipe = Pipeline([
    ("pre", pre),
    ("model", model)
])

# entrenar modelo
pipe.fit(X_train, y_train)

# predicciones
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:,1]

# accuracy
acc = accuracy_score(y_test, y_pred)
print("accuracy en test:", acc)

# matriz de confusion
cm = confusion_matrix(y_test, y_pred)
print("matriz de confusion")
print(cm)

# reporte de clasificacion
print("classification report")
print(classification_report(y_test, y_pred))

# auc
auc = roc_auc_score(y_test, y_prob)
print("auc:", auc)

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("curva roc")
plt.show()

# validacion cruzada
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

acc_cv = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
auc_cv = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")

print("accuracy cv de cada fold")
print(acc_cv)
print("accuracy cv promedio:", acc_cv.mean())

print("auc cv de cada fold")
print(auc_cv)
print("auc cv promedio:", auc_cv.mean())

# score final
print("score final del modelo:", pipe.score(X_test, y_test))